# Adjacency Representations

```{contents}
:local:
:depth: 2
```

```{index} Adjacency Matrices
```

## Adjacency Matrices




```{index} adjacency matrix; vertex index; symmetry
```

An adjacency matrix reserves one cell for each ordered pair of vertices. A row describes outgoing connections from its vertex; a column describes incoming connections to its vertex. Map symbolic labels to dense integer indexes before accessing the array.

For the six-room undirected graph, a Boolean matrix has 36 cells, although only eight are true. Four logical edges appear twice, once in each direction. Its diagonal is false because the model rejects self-loops. An isolated vertex still has a row and column, both empty.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] rooms = { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" };
var index = rooms.Select((room, i) => (room, i)).ToDictionary(x => x.room, x => x.i);
bool[,] connected = new bool[rooms.Length, rooms.Length];
foreach (var (a, b) in new[] { ("Lobby", "Hall"), ("Hall", "Lab"), ("Hall", "Office"), ("Office", "Storage") })
{
    connected[index[a], index[b]] = true;
    connected[index[b], index[a]] = true;
}
foreach (string room in rooms)
{
    int row = index[room];
    Console.WriteLine($"{room}: {string.Join(" ", Enumerable.Range(0, rooms.Length).Select(col => connected[row, col] ? 1 : 0))}");
}

Lobby: 0 1 0 0 0 0
Hall: 1 0 1 1 0 0
Lab: 0 1 0 0 0 0
Office: 0 1 0 0 1 0
Storage: 0 0 0 1 0 0
Remote: 0 0 0 0 0 0


The Hall row is `1 0 1 1 0 0`; the Remote row is all zeros. In an undirected matrix, `[i, j]` equals `[j, i]`. A directed matrix need not be symmetric. Do not infer a reverse arc merely because an undirected example wrote both cells.

Once indexes are known, direct edge lookup reads one cell in O(1) time. Enumerating a vertex's neighbors scans its entire row, Θ(V) time. Adding or removing vertices in a tightly sized matrix can require a new array and remapping many indexes. Keeping spare capacity or inactive indexes changes that tradeoff and needs its own policy.

**Common mistake.** Setting only `connected[index[a], index[b]]` for an undirected edge. The matrix then says Hall reaches Lab but Lab does not reach Hall, and any traversal that starts at Lab reports a different graph.


```{index} Weights and Absence
```

## Weights and Absence




```{index} weighted adjacency matrix; nullable weight; zero-weight edge
```

A nullable weight matrix separates a missing edge (`null`) from an existing edge with weight zero. Initializing every cell to zero and treating zero as absence would erase valid zero-minute routes. The diagonal remains null under this simple-graph contract; a zero-cost path from a vertex to itself does not imply a stored self-loop.

In [1]:
using System;
int?[,] minutes = new int?[3, 3];
minutes[0, 1] = minutes[1, 0] = 3;
minutes[1, 2] = minutes[2, 1] = 0;
Console.WriteLine($"edge 1-2 exists={minutes[1, 2].HasValue}, weight={minutes[1, 2]}");
Console.WriteLine($"edge 0-2 exists={minutes[0, 2].HasValue}");
Console.WriteLine($"stored loop 0-0={minutes[0, 0].HasValue}");

edge 1-2 exists=True, weight=0
edge 0-2 exists=False
stored loop 0-0=False


The output is `edge 1-2 exists=True, weight=0`, `edge 0-2 exists=False`, and `stored loop 0-0=False`. The zero-minute route between rooms 1 and 2 is a real edge, distinguishable from the absent 0-2 edge only because the cell type is `int?`; the diagonal stays `null` because the model stores no self-loops.

**Common mistake.** Declaring the matrix as `int[,]` and using zero for a missing edge. The zero-minute Office to Storage route then reads as no route at all, and a shortest-path search routes around a corridor that exists.


```{index} Adjacency Collections
```

## Adjacency Collections




```{index} adjacency list; neighbor map; sparse graph
```

An adjacency list stores each vertex's actual neighbors. The term includes several concrete choices: a list, a set, or a dictionary of neighbor-to-weight pairs. These choices have different lookup and update costs.

A list supports neighbor iteration but must scan to find a particular neighbor, and it can contain duplicates unless insertion checks for them. A hash set supports expected constant-time membership for unweighted edges. A neighbor dictionary supports expected constant-time lookup and weight replacement while keeping one entry per neighbor. All retain explicit entries for isolated vertices.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var adjacency = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal);
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Remote" })
    adjacency.Add(room, new Dictionary<string, int>(StringComparer.Ordinal));
adjacency["Lobby"]["Hall"] = adjacency["Hall"]["Lobby"] = 4;
adjacency["Hall"]["Lab"] = adjacency["Lab"]["Hall"] = 2;
foreach (string room in adjacency.Keys.OrderBy(r => r, StringComparer.Ordinal))
    Console.WriteLine($"{room}: {string.Join(", ", adjacency[room].Keys.OrderBy(r => r, StringComparer.Ordinal))}");
Console.WriteLine($"Hall-Lab weight={adjacency["Hall"]["Lab"]}");

Hall: Lab, Lobby
Lab: Hall
Lobby: Hall
Remote: 
Hall-Lab weight=2


In this undirected model, each logical edge has two stored adjacency entries. The space bound is Θ(V + E), since doubling E changes only the constant factor. A directed outgoing adjacency representation stores one entry per arc. Forgetting the reverse entry in an undirected update creates an asymmetric representation of a supposedly symmetric relationship.

Sorting the keys above is a reporting choice, not a hash-table guarantee. The underlying neighbor dictionary does not promise numerical or alphabetical traversal order. Later graph algorithms may produce different valid visit orders when neighbor enumeration order differs.

**Common mistake.** Forgetting the reverse entry in an undirected update, writing `adjacency["Hall"]["Lab"] = 2` alone. The graph becomes directed by accident; `IsValid`-style symmetry checks, as in the protected graph below, catch exactly this.


```{index} Guided Practice: List a Room's Neighbors
```

### Guided Practice: List a Room's Neighbors


Print the neighbors of `Hall` in alphabetical order from the adjacency collection, and then the number of neighbors `Remote` has. Expected output: `Lab, Lobby` and `0`.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var adjacency = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal);
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Remote" })
    adjacency.Add(room, new Dictionary<string, int>(StringComparer.Ordinal));
adjacency["Lobby"]["Hall"] = adjacency["Hall"]["Lobby"] = 4;
adjacency["Hall"]["Lab"] = adjacency["Lab"]["Hall"] = 2;

// Your code starts here.
Console.WriteLine(string.Join(", ", adjacency.Keys));
// Your code ends here.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
var adjacency = new Dictionary<string, Dictionary<string, int>>(StringComparer.Ordinal);
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Remote" })
    adjacency.Add(room, new Dictionary<string, int>(StringComparer.Ordinal));
adjacency["Lobby"]["Hall"] = adjacency["Hall"]["Lobby"] = 4;
adjacency["Hall"]["Lab"] = adjacency["Lab"]["Hall"] = 2;

var neighbors = adjacency["Hall"].Keys.OrderBy(name => name, StringComparer.Ordinal);
Console.WriteLine(string.Join(", ", neighbors));
Console.WriteLine(adjacency["Remote"].Count);


Lab, Lobby
0


The inner dictionary for a room *is* its neighbor list, so the lookup costs one hash access plus the size of that list, not a scan of every room. `Remote` has an entry with zero neighbors; a missing key would instead throw, which is why vertices are added before any edge.


```{index} Density and Workload Costs
```

## Density and Workload Costs




```{index} graph density; dense graph; representation cost
```

For a simple undirected graph with V vertices, the maximum number of edges is `V(V - 1) / 2`: choose an unordered pair of distinct endpoints. The directed maximum without loops is `V(V - 1)` because each pair has two possible directions. Density is actual edges divided by the relevant maximum. For fewer than two vertices, this chapter reports density zero to avoid division by zero.

| Operation | Matrix with known indexes | List per vertex | Hash neighbor map |
| --- | --- | --- | --- |
| Storage | Θ(V²) | Θ(V + E) | Θ(V + E) |
| Direct edge lookup | O(1) | O(deg(u)) | Expected O(1) |
| Iterate neighbors of u | Θ(V) | Θ(deg(u)) | Θ(deg(u)) |
| Set edge weight | O(1) | O(deg(u)) to find it | Expected amortized O(1) |
| Add an isolated vertex | May require Θ(V²) copying | Amortized O(1) in a growable indexed collection | Expected amortized O(1) |

Hash bounds assume well-distributed hashing and fixed-cost label operations; growth can make an individual insertion expensive. Label-to-index lookup is additional work for the matrix. The table's update costs describe directed entries; an undirected edge updates two entries with the same asymptotic bound.

In [1]:
using System;
int vertices = 50, edges = 60;
long possible = (long)vertices * (vertices - 1) / 2;
double density = possible == 0 ? 0 : (double)edges / possible;
Console.WriteLine($"possible edges={possible}, density={density:P2}");
Console.WriteLine($"matrix cells={(long)vertices * vertices}, undirected adjacency entries={2 * edges}");

possible edges=1225, density=4.90%
matrix cells=2500, undirected adjacency entries=120


This graph has 60 of 1225 possible edges, about 4.90% density. A matrix reserves 2500 pair cells; adjacency collections store 120 neighbor entries plus 50 vertex entries. These are counts of storage locations, not equal-size bytes. Dictionary objects, hashing, and arrays have different overheads, so small graphs may favor a matrix even when sparse.

Choose by operations as well as density. Frequent pair checks with a fixed dense vertex set favor a matrix; scanning neighbors in a large sparse graph favors adjacency collections. No fixed density cutoff is a universal decision rule.

**Common mistake.** Choosing a matrix for a sparse graph because lookup is O(1). With 50 vertices and 60 edges the matrix reserves 2,500 cells for 120 entries, and every neighbor scan still reads a whole row of mostly empty cells.


```{index} Code Tracing: Read the Matrix as a Graph
```

### Code Tracing: Read the Matrix as a Graph

The directed graph below is stored as a matrix in which row r, column c is `true` when an arc runs from vertex r to vertex c. Without running the solution, list each vertex's out-neighbors and in-degree, decide whether the graph has a directed cycle, and name the vertices with in-degree zero.

```text
        A  B  C  D  E
    A   0  1  1  0  0
    B   0  0  0  1  0
    C   0  0  0  1  0
    D   0  0  0  0  1
    E   0  0  0  0  0
```

In [ ]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

string[] names = { "A", "B", "C", "D", "E" };
bool[,] arc =
{
    { false, true,  true,  false, false },
    { false, false, false, true,  false },
    { false, false, false, true,  false },
    { false, false, false, false, true  },
    { false, false, false, false, false },
};
int n = names.Length;
for (int r = 0; r < n; r++)
{
    var outs = Enumerable.Range(0, n).Where(c => arc[r, c]).Select(c => names[c]);
    int indegree = Enumerable.Range(0, n).Count(p => arc[p, r]);
    Console.WriteLine($"{names[r]}: out -> [{string.Join(", ", outs)}]  in-degree {indegree}");
}
Console.WriteLine($"in-degree zero: {string.Join(", ", Enumerable.Range(0, n).Where(r => Enumerable.Range(0, n).All(p => !arc[p, r])).Select(r => names[r]))}");
Console.WriteLine($"symmetric (undirected)? {Enumerable.Range(0, n).All(r => Enumerable.Range(0, n).All(c => arc[r, c] == arc[c, r]))}");

A: out -> [B, C]  in-degree 0
B: out -> [D]  in-degree 1
C: out -> [D]  in-degree 1
D: out -> [E]  in-degree 2
E: out -> []  in-degree 1
in-degree zero: A
symmetric (undirected)? False


Row A reads B and C; rows B and C each read D; row D reads E; row E is empty. Column counts give in-degrees 0, 1, 1, 2, 1. The matrix is not symmetric, so the graph is directed, and it has no directed cycle: every arc goes from an earlier letter to a later one, so no walk can return to its start. Only A has in-degree zero, which makes it the one place a topological order can begin, as 20.3 shows. A reader who reported the cycle A, B, D, E, A has read column E as a row.

```{index} Code Completion: Build the Matrix from the Neighbor Map
```

### Code Completion: Build the Matrix from the Neighbor Map

Convert the undirected neighbor map into a Boolean matrix. Complete the loop so that each stored neighbor sets the right cell; because the map already holds both directions of every edge, one assignment per entry is enough. The expected output is the four matrix rows followed by `symmetric: True`.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

string[] rooms = { "Lobby", "Hall", "Lab", "Remote" };
var index = rooms.Select((room, i) => (room, i)).ToDictionary(x => x.room, x => x.i);
var adjacency = new Dictionary<string, HashSet<string>>
{
    ["Lobby"] = new() { "Hall" }, ["Hall"] = new() { "Lobby", "Lab" }, ["Lab"] = new() { "Hall" }, ["Remote"] = new(),
};

bool[,] connected = new bool[rooms.Length, rooms.Length];
foreach (var (room, neighbors) in adjacency)
{
    foreach (string neighbor in neighbors)
    {
        // Your code starts here.

        // Your code ends here.
    }
}

foreach (string room in rooms)
    Console.WriteLine($"{room,-7} {string.Join(" ", rooms.Select(other => connected[index[room], index[other]] ? 1 : 0))}");
bool symmetric = rooms.All(a => rooms.All(b => connected[index[a], index[b]] == connected[index[b], index[a]]));
Console.WriteLine($"symmetric: {symmetric}");

In [ ]:
// Solution
using System;
using System.Collections.Generic;
using System.Linq;

string[] rooms = { "Lobby", "Hall", "Lab", "Remote" };
var index = rooms.Select((room, i) => (room, i)).ToDictionary(x => x.room, x => x.i);
var adjacency = new Dictionary<string, HashSet<string>>
{
    ["Lobby"] = new() { "Hall" }, ["Hall"] = new() { "Lobby", "Lab" }, ["Lab"] = new() { "Hall" }, ["Remote"] = new(),
};

bool[,] connected = new bool[rooms.Length, rooms.Length];
foreach (var (room, neighbors) in adjacency)
{
    foreach (string neighbor in neighbors)
    {
        connected[index[room], index[neighbor]] = true;   // the reverse entry sets the mirrored cell
    }
}

foreach (string room in rooms)
    Console.WriteLine($"{room,-7} {string.Join(" ", rooms.Select(other => connected[index[room], index[other]] ? 1 : 0))}");
bool symmetric = rooms.All(a => rooms.All(b => connected[index[a], index[b]] == connected[index[b], index[a]]));
Console.WriteLine($"symmetric: {symmetric}");

Lobby   0 1 0 0
Hall    1 0 1 0
Lab     0 1 0 0
Remote  0 0 0 0
symmetric: True


One assignment per stored entry is correct here only because the neighbor map stores each undirected edge twice; converting a directed map would use the same line and produce an asymmetric matrix. The `Remote` row and column stay all zeros, and the symmetry check is the same invariant the protected graph below enforces. The conversion costs Θ(V + E) to walk the map plus Θ(V²) to allocate the matrix, which is the space trade the density discussion describes.

```{index} A Protected Route Graph
```

## A Protected Route Graph




```{index} graph API; undirected invariant; defensive copy
```

Encapsulation should protect graph rules, not merely hide a field's name. `RouteGraph` uses private neighbor dictionaries and returns copied arrays for vertices and neighbors. A caller can edit its copy without changing the stored graph. Returning a mutable collection as `IEnumerable` would not protect it: a caller could cast it back to the original collection type.

The contract requires explicit vertices. `SetEdge` checks both endpoints and weight rules before changing either direction. A new logical edge increases EdgeCount once; replacing its weight changes both reciprocal entries without changing EdgeCount. `TryGetWeight` returns false for an absent endpoint or edge, while degree/neighbor queries throw for an unknown vertex so it cannot masquerade as an isolate.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");
Console.WriteLine($"replacement inserted={graph.SetEdge("Hall", "Lab", 5)}, edges={graph.EdgeCount}");
Console.WriteLine($"Hall-Lab found={graph.TryGetWeight("Hall", "Lab", out int minutes)}, minutes={minutes}");
var snapshot = graph.Neighbors("Hall");
snapshot[0] = new KeyValuePair<string, int>("Injected", 999);
Console.WriteLine($"snapshot changed graph={graph.TryGetWeight("Hall", "Injected", out _)}");
Console.WriteLine($"Remote degree={graph.Degree("Remote")}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

vertices=6, edges=4, valid=True
replacement inserted=False, edges=4
Hall-Lab found=True, minutes=5
snapshot changed graph=False
Remote degree=0


The output is `vertices=6, edges=4, valid=True`, then `replacement inserted=False, edges=4`, then `Hall-Lab found=True, minutes=5`, then `snapshot changed graph=False`, and `Remote degree=0`. Reassigning Hall to Lab changed the weight in both directions without adding an edge, and the edited snapshot left the graph untouched because `Neighbors` returned a copy.

**Common mistake.** Returning the internal dictionary as `IEnumerable<KeyValuePair<string, int>>` and calling it read-only. A caller can cast it back and modify the graph behind the class's checks; the copy returned by `Neighbors` is what the `snapshot` lines in the cell prove.


```{index} Removal, Invariants, and Testing
```

## Removal, Invariants, and Testing




```{index} edge removal; vertex removal; degree sum; graph validation
```

Removing an undirected edge deletes both reciprocal entries and reduces EdgeCount once. Removing a vertex deletes every reverse reference before removing its own dictionary. Its degree tells you how many logical edges disappear. Other vertices stay present even if they become isolated.

Trace removing Hall from the six-room example. Hall's neighbors are Lobby, Lab, and Office. Remove Hall from each of those neighbor maps, subtract three from EdgeCount, and delete Hall. Five vertices and one edge remain; Lobby and Lab are now isolates, Office–Storage remains, and Remote is unchanged. Removing Hall again returns false.

The representation invariant requires every neighbor label to name an existing vertex, no self-loops, nonnegative weights, equal reverse weights, and degree sum `2 * EdgeCount`. `IsValid` checks those rules in expected Θ(V + E) time. It still cannot prove that the data matches the intended floor plan, so compare edge membership and weights with an independent model too.

With neighbor hashes, `RemoveEdge` takes expected O(1) time. `RemoveVertex(u)` takes expected O(deg(u)) time because it removes a reference at each neighbor. The code does not mutate u's neighbor map while enumerating it; it changes the distinct maps belonging to those neighbors. Self-loops are rejected, so a reverse map cannot be the same map being enumerated.

`Degree` reads a stored dictionary count in expected O(1). `Neighbors` copies deg(u) entries, so it takes Θ(deg(u)) time and space. `Vertices` copies V labels. Defensive copying has a cost, but keeps callers from bypassing update rules.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A social network has 50 million users and about 100 friendships per user. Which representation, and roughly how much is saved?

```{dropdown} Answer
Adjacency lists. The matrix would need 50 million squared cells, 2.5 quadrillion, almost all empty; the lists store about 5 billion entries, two per friendship. The graph is extremely sparse, and neighbor iteration, the common operation, reads only a user's own list.
```

2. When is the adjacency matrix the better choice despite its V² storage?

```{dropdown} Answer
When V is small or the graph is dense, and the dominant operation is asking whether a specific pair is connected: a matrix answers in one array read with no hashing. A fixed vertex set also removes the matrix's weakness at adding vertices.
```

3. `SetEdge("Hall", "Lab", 5)` on an existing edge returned `false`. What changed in the graph?

```{dropdown} Answer
The weight in both stored directions changed to 5 and nothing else: `EdgeCount` is unchanged because no new logical edge was created. The `false` return reports replacement, matching the dictionary convention from Chapter 19.
```

4. Why does the route graph return a copied array from `Neighbors` rather than the internal dictionary?

```{dropdown} Answer
So that the class's rules cannot be bypassed. A caller holding the real dictionary could add a neighbor on one side only, breaking symmetry and the degree-sum invariant, without `SetEdge` ever running its checks. The copy costs Θ(deg(u)) and keeps every update inside the class.
```

```{index} Exercise
```

## Exercise




```{index} undirected edge removal; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: remove both stored directions of one undirected edge.
var adjacency = new Dictionary<string, Dictionary<string, int>>
{
    ["A"] = new() { ["B"] = 3 },
    ["B"] = new() { ["A"] = 3, ["C"] = 0 },
    ["C"] = new() { ["B"] = 0 }
};
// Your code starts here.
/* TODO: remove A-B in both directions while preserving B-C. */
// Your code ends here.
Console.WriteLine($"A-B={adjacency["A"].ContainsKey("B")}, B-A={adjacency["B"].ContainsKey("A")}");
Console.WriteLine($"B-C={adjacency["B"]["C"]}, C-B={adjacency["C"]["B"]}");

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
var adjacency = new Dictionary<string, Dictionary<string, int>>
{
    ["A"] = new() { ["B"] = 3 },
    ["B"] = new() { ["A"] = 3, ["C"] = 0 },
    ["C"] = new() { ["B"] = 0 }
};
adjacency["A"].Remove("B"); adjacency["B"].Remove("A");
Console.WriteLine($"A-B={adjacency["A"].ContainsKey("B")}, B-A={adjacency["B"].ContainsKey("A")}");
Console.WriteLine($"B-C={adjacency["B"]["C"]}, C-B={adjacency["C"]["B"]}");

A-B=False, B-A=False
B-C=0, C-B=0


The output is `A-B=False, B-A=False` and `B-C=0, C-B=0`. Both stored directions of the A-B edge are gone, and the zero-weight B-C edge survived in both directions; removing only `adjacency["A"]["B"]` would have left a one-way edge that the symmetry check rejects.


```{rubric} Footnotes
```


[^2002_adjacency_representations-1]: Directed graphs may also maintain incoming adjacency collections. That uses additional storage but makes removal of incoming arcs and reverse-neighbor queries faster.
[^2002_adjacency_representations-2]: Representation costs depend on the concrete neighbor collection; an adjacency list implemented with hash maps differs from one implemented with ordinary lists.